In [67]:
import os
import json
import requests
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd

# Preparing and prepping datafiles for merging
1. Annotinder files

In [68]:
file_names = [
    "annotations_95_AnnoBias job final, set 1.csv.csv",
    "annotations_96_AnnoBias job final, set 2.csv.csv",
    "annotations_97_AnnoBias job final, set 3.csv.csv",
    "annotations_98_AnnoBias job final, set 4.csv.csv",
    "annotations_99_AnnoBias job final, set 5.csv.csv",
    "annotations_100_AnnoBias job final, set 6.csv.csv"
]

dataframes = []

for idx, file in enumerate(file_names, start=1):
    df = pd.read_csv(f"/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/AnnoTinder_data_exports/{file}")  
    
    # Add a "condition" column based on the file number
    df['condition_AT'] = idx    
    dataframes.append(df)  # Append the DataFrame to the list

annotinder_df = pd.concat(dataframes, ignore_index=True)
annotinder_df['uid'] = annotinder_df['coder']

2. Twitter data

In [69]:
X_data = pd.read_csv('/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/X_data/final_stratified_immigration_with_stances.csv')
X_data['tweet_id'] = X_data['tweet_id'].astype(str)

3. Qualtrics data 

In [70]:
# Define the cutoff date. Delete trail data from before the Soft Launch
cutoff_date = datetime(2024, 10, 14)

# Load and process the "before_before" dataset
before_before = pd.read_csv('/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/Qualtrics_data_exports/start_survey.csv', skiprows=[1, 2])
#print(f"Total rows in before_before: {len(before_before)}")

# Convert StartDate to datetime and filter based on cutoff_date
before_before["time_question"] = pd.to_datetime(before_before["StartDate"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
before_before = before_before[before_before["time_question"] >= cutoff_date]

print(f"{len(before_before)} participants started the experiment")

# Load and process the "before" dataset
before = pd.read_csv('/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/Qualtrics_data_exports/before.csv', skiprows=[1, 2])
#print(f"Total rows in before: {len(before)}")

before["time_question"] = pd.to_datetime(before["StartDate"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
before = before[before["time_question"] >= cutoff_date]

print(f"{len(before)} participants agreed to informed consent")

# Load and process the "after" dataset
after = pd.read_csv('/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/Qualtrics_data_exports/after.csv', skiprows=[1, 2])
#print(f"Total rows in after: {len(after)}")

after["time_question"] = pd.to_datetime(after["StartDate"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
after = after[after["time_question"] >= cutoff_date]

print(f"{len(after)} continued after IAT")

2338 participants started the experiment
2163 participants agreed to informed consent
1584 continued after IAT


In [71]:
before_after = pd.merge(before, after, on='uid', how='right')
before_before['StartDate_First'] = before_before['StartDate']
qualtrics_merged = pd.merge(before_before, before_after, on='psid')

print(f"for {len(qualtrics_merged)} participants, the datasets could be merged")
# Keep only the first observation for each 'uid'
qualtrics_merged = qualtrics_merged.drop_duplicates(subset=['uid'], keep='first')

print(f"After ensuring only one observation per uid, {len(qualtrics_merged)} remain")
print(f"Unique uids after deduplication: {len(set(qualtrics_merged['uid']))}")

for 1560 participants, the datasets could be merged
After ensuring only one observation per uid, 1543 remain
Unique uids after deduplication: 1543


In [72]:
# Remove rows with missing 'dsc' values
qualtrics_merged = qualtrics_merged.dropna(subset=['dsc'])
print(f"After removing participants for whom no dsc scores could be calculated, {len(qualtrics_merged)} remain")

After removing participants for whom no dsc scores could be calculated, 1443 remain


Merge AnnoTinder and Twitter data and Qualtrics data

In [73]:
from datetime import datetime
import pandas as pd

# Define the cutoff date for filtering time_question
cutoff_date = datetime(2024, 10, 14)

# Step 1: Function to handle non-numeric values and trim the first 10 digits
def trim_unit_id(unit_id):
    try:
        # Try to convert to float, then to int, and slice the first 10 characters
        return str(int(float(unit_id)))[0:10]
    except ValueError:
        # Return the original value if it's non-numeric (handle as is)
        return unit_id

# Optionally, check the unique values to confirm '50PLUS' and 'BIJ1' are no longer present
print(f"Remaining parties in 'normalized_party': {X_data['normalized_party'].unique()}")

# Step 2: Apply the trim_unit_id function to 'unit_id' in annotinder_df and 'tweet_id' in X_data
annotinder_df['unit_id_15'] = annotinder_df['unit_id'].apply(trim_unit_id)
X_data['tweet_id_15'] = X_data['tweet_id'].apply(trim_unit_id)

# Step 3: Filter X_data to include only rows with matching tweet_id_15 in annotinder_df
matching_rows = X_data[X_data['tweet_id_15'].isin(annotinder_df['unit_id_15'])]

# Step 4: Merge all columns from X_data with annotinder_df based on tweet_id_15 and unit_id_15
annotinder_twitter = annotinder_df.merge(
    matching_rows, 
    left_on='unit_id_15', right_on='tweet_id_15', 
    how='left'
)

# Step 5: Ensure 'time_question' column is converted to datetime (if necessary)
if annotinder_twitter['time_question'].dtype != 'datetime64[ns]':
    annotinder_twitter['time_question'] = pd.to_datetime(
        annotinder_twitter['time_question'], 
        format="%Y-%m-%dT%H:%M:%S.%fZ", 
        errors='coerce'
    )

# Step 6: Filter annotinder_twitter to only include rows where time_question >= cutoff_date
annotinder_twitter = annotinder_twitter[annotinder_twitter['time_question'] >= cutoff_date]

# Step 7: Merge with qualtrics_merged DataFrame on 'uid'
if 'uid' in annotinder_twitter.columns and 'uid' in qualtrics_merged.columns:
    df = pd.merge(
        annotinder_twitter, 
        qualtrics_merged, 
        on='uid', 
        suffixes=('_annotinder', '_qualtrics')
    )
    print(f"After applying the cutoff date and merging, {len(df)} rows remain.")
else:
    print("Warning: 'uid' column not found in both DataFrames. Please check the column names.")


Remaining parties in 'normalized_party': ['D66' 'Volt' 'CDA' 'ChristenUnie' '50PLUS' 'Nieuw Sociaal Contract' 'SP'
 'GroenLinks' 'BIJ1' 'Partij voor de Dieren' 'PvdA' 'DENK'
 'GroenLinks-PvdA' 'PVV' 'Forum voor Democratie' 'JA21' 'VVD' 'SGP']
After applying the cutoff date and merging, 156346 rows remain.


In [ ]:
#annotinder_twitter[['time_answer','text', 'source', 'uid']].head(50)

In [74]:
print(f"for {df['uid'].nunique()} participants, the AnnoTinder, qualtrics and Twitter datasets could be merged")

for 1358 participants, the AnnoTinder, qualtrics and Twitter datasets could be merged


In [75]:
# Check and remove tweets from BIJ1 and 50PLUS as we do not have measurements for these parties:
print(f"Number of Annotations from BIJ1: {len(df[df['normalized_party'] == 'BIJ1'])}")
print(f"Number of Annotations from 50PLUS: {len(df[df['normalized_party'] == '50PLUS'])}")

Number of Annotations from BIJ1: 1632
Number of Annotations from 50PLUS: 1040


In [76]:
## Removing anntoations from BIj1 and 50plus:

# Remove rows from df where normalized_party is 'BIJ1' or '50PLUS'
print(f"Size of the dataset before removal: {len(df)}")

# Filter out rows with normalized_party as 'BIJ1' or '50PLUS'
df = df[~df['normalized_party'].isin(['BIJ1', '50PLUS'])]

print(f"Size of the dataset after removal: {len(df)}")

Size of the dataset before removal: 156346
Size of the dataset after removal: 153674


In [77]:
# Function to calculate time differences
def calculate_time_differences_fixed_mdt(start_dates, time_answers):
    differences = []
    
    for start, answer in zip(start_dates, time_answers):
        # Ensure both start and answer are valid
        if pd.isnull(start) or pd.isnull(answer):
            differences.append(None)  # Missing data
            continue
        
        try:
            # Parse StartDate_First (MDT: UTC-6)
            start_dt = datetime.strptime(str(start), '%Y-%m-%d %H:%M:%S')
            start_dt_utc = (start_dt + timedelta(hours=6)).replace(tzinfo=timezone.utc)
            
            # Parse time_answer (UTC)
            answer_dt = datetime.strptime(str(answer), '%Y-%m-%dT%H:%M:%S.%fZ').replace(tzinfo=timezone.utc)
            
            # Calculate time difference in minutes
            time_diff = (answer_dt - start_dt_utc).total_seconds() / 60
            differences.append(time_diff)
        except Exception as e:
            print(f"Error processing row: {start}, {answer} - {e}")
            differences.append(None)
    
    return differences

# Calculate time differences
time_differences = calculate_time_differences_fixed_mdt(df['StartDate_First'], df['time_answer'])

# Add time_difference column to DataFrame
df['time_difference'] = time_differences

# Sort and get the last observation for each psid
df = df.sort_values(by=['psid', 'time_answer'])
last_observation = df.groupby('psid').last().reset_index()

In [78]:
df.tail(5)

,coder_id,coder,jobset,unit_id,unit_status,variable,value,time_question_annotinder,time_answer,field,...,Q6_Page Submit,Q6_Click Count,sid_y,project_y,dsc,dst,Condition,IP_Address_y,time_question_y,time_difference
107929,14129,266261,batch_29,1751994181583606016,DONE,stellingen.misinformation,Neutraal,2024-10-15 09:30:32.109,2024-10-15T09:30:45.238Z,NaN,...,NaN,NaN,8554_5150,IAT_2024,0.51324,1.0,5,31.20.84.125,2024-10-15 03:27:43,10.553967
107942,14129,266261,batch_29,1369999670588624896,DONE,stellingen.political_stance,Gedeeltelijk waar,2024-10-15 09:30:46.666,2024-10-15T09:30:47.295Z,NaN,...,NaN,NaN,8554_5150,IAT_2024,0.51324,1.0,5,31.20.84.125,2024-10-15 03:27:43,10.588250
107943,14129,266261,batch_29,1369999670588624896,DONE,stellingen.sentiment,Gedeeltelijk niet waar,2024-10-15 09:30:46.666,2024-10-15T09:30:47.429Z,NaN,...,NaN,NaN,8554_5150,IAT_2024,0.51324,1.0,5,31.20.84.125,2024-10-15 03:27:43,10.590483
107944,14129,266261,batch_29,1369999670588624896,DONE,stellingen.toxic,Gedeeltelijk waar,2024-10-15 09:30:46.666,2024-10-15T09:30:47.580Z,NaN,...,NaN,NaN,8554_5150,IAT_2024,0.51324,1.0,5,31.20.84.125,2024-10-15 03:27:43,10.593000
107945,14129,266261,batch_29,1369999670588624896,DONE,stellingen.misinformation,Neutraal,2024-10-15 09:30:46.666,2024-10-15T09:30:47.810Z,NaN,...,NaN,NaN,8554_5150,IAT_2024,0.51324,1.0,5,31.20.84.125,2024-10-15 03:27:43,10.596833


## Compute variables

In [79]:
df['uid'].nunique()

1358

In [80]:
pd.crosstab(df['condition_AT'], df['Condition'])

Condition,1,2,3,4,5,6
condition_AT,,,,,,
1,25303,0,0,0,0,0
2,0,27211,0,0,0,0
3,0,0,25150,0,0,0
4,0,0,0,25593,0,0
5,0,0,0,0,25100,0
6,0,0,0,0,0,25317


In [81]:
df['uid'].nunique()
df['uid'].nunique()
df[['uid', 'jobset']].nunique()

uid       1358
jobset      62
dtype: int64

In [82]:
print(df['normalized_party'].unique())

[nan 'SP' 'GroenLinks' 'Partij voor de Dieren' 'PvdA' 'DENK'
 'GroenLinks-PvdA' 'VVD' 'PVV' 'Forum voor Democratie' 'JA21' 'Volt' 'D66'
 'CDA' 'ChristenUnie' 'Nieuw Sociaal Contract' 'SGP']


In [83]:
# Sample likert_mapping (you have already defined this)
likert_mapping = {
    "Geheel onwaarschijnlijk": 1,
    "Onwaarschijnlijk": 2,
    "Niet waarschijnlijk, maar ook niet onwaarschijnlijk": 3,
    "Waarschijnlijk": 4,
    "Zeer waarschijnlijk": 5
}

# Assuming df is your DataFrame containing individual data

# Step 1: Map Likert responses to numeric values for each party
party_columns = ['PolCongr_1', 'PolCongr_2', 'PolCongr_3', 'PolCongr_4', 'PolCongr_5', 
                 'PolCongr_6', 'PolCongr_7', 'PolCongr_8', 'PolCongr_9', 'PolCongr_10',
                 'PolCongr_11', 'PolCongr_12', 'PolCongr_13', 'PolCongr_14', 'PolCongr_15']

# Map Likert scale responses to numeric values for all relevant columns
for party in party_columns:
    df[party] = df[party].map(likert_mapping)

# Step 2: Create a mapping from party names to their corresponding PolCongr_* columns
party_to_column_mapping = {
    'PVV': 'PolCongr_1',
    'GroenLinks-PvdA': 'PolCongr_2',  # GL-PvdA combined
    'VVD': 'PolCongr_3',
    'Nieuw Sociaal Contract': 'PolCongr_4',
    'D66': 'PolCongr_5',
    'BBB': 'PolCongr_6',
    'CDA': 'PolCongr_7',
    'SP': 'PolCongr_8',
    'DENK': 'PolCongr_9',
    'Partij voor de Dieren': 'PolCongr_10',
    'Forum voor Democratie': 'PolCongr_11',
    'SGP': 'PolCongr_12',
    'ChristenUnie': 'PolCongr_13',
    'Volt': 'PolCongr_14',
    'JA21': 'PolCongr_15'
}


# Step 3: Combine the relevant parties ('GroenLinks', 'PvdA', 'GroenLinks-PvdA') into a single group
df['normalized_party'] = df['normalized_party'].replace({
    'GroenLinks': 'GroenLinks-PvdA',  # Combine GroenLinks and PvdA into GroenLinks-PvdA
    'PvdA': 'GroenLinks-PvdA'
})


# Step 4: Calculate the score for the normalized party (single numeric score)

def calculate_party_score(row):
    normalized_party = row['normalized_party']
    
    # Look up the column corresponding to the normalized party
    normalized_party_column = party_to_column_mapping.get(normalized_party)
    
    if normalized_party_column:
        # Return the score from the corresponding column for the normalized party
        return row[normalized_party_column]
    else:
        return None

# Step 6: Apply the function to calculate the score for each participant
df['vote_likelihood_score'] = df.apply(calculate_party_score, axis=1)

# Step 7: Check the updated DataFrame with the vote likelihood score
#df[['normalized_party', 'vote_likelihood_score']].tail(50)
df['vote_likelihood_score'].value_counts()

df[['normalized_party', 'vote_likelihood_score']].tail(3)

,normalized_party,vote_likelihood_score
107943,SP,2.0
107944,SP,2.0
107945,SP,2.0


In [84]:
df.groupby('normalized_party')['dsc'].mean().sort_values(ascending=False)

normalized_party
Partij voor de Dieren     0.548843
GroenLinks-PvdA           0.529799
DENK                      0.528117
SP                        0.514232
Forum voor Democratie     0.495250
ChristenUnie              0.491859
PVV                       0.491446
VVD                       0.490319
CDA                       0.488963
JA21                      0.483937
Volt                      0.476722
D66                       0.473886
Nieuw Sociaal Contract    0.467244
SGP                       0.457130
Name: dsc, dtype: float64

In [85]:
# Define the mapping for instructions
instruction_mapping = {
    1: "no instructions",
    2: "general instructions",
    3: "tailored instructions",
    4: "no instructions",
    5: "general instructions",
    6: "tailored instructions",
}

# Convert condition to integers before mapping
df["instruction_type"] = df["Condition"].astype(int).map(instruction_mapping)

# Define whether the condition includes author handle
df["source_shown"] = df["Condition"].astype(int).apply(lambda x: 0 if x in [1, 2, 3] else 1)

# Assuming df['Age_y'] contains the year of birth
current_year = datetime.now().year

# Create the 'Age' column by subtracting the year of birth from the current year
df['Age'] = current_year - df['Age_qualtrics']
df['Male'] = df['gender'].map({'male': 1, 'female': 0, 'other/unknown': 0})

In [87]:
df['instruction_type'].value_counts()
df['source_shown'].value_counts()

source_shown
0    77664
1    76010
Name: count, dtype: int64

In [88]:
# Maak een nieuwe kolom 'Etniciteit_binair'
df['Etniciteit_binair'] = df['Etniciteit'].apply(lambda x: 1 if x == 'Nederlands' else 0)
df['Etniciteit_binair'].value_counts()

Etniciteit_binair
1    140523
0     13151
Name: count, dtype: int64

In [89]:
df['Werk'].value_counts()

Werk
Werkend (betaalde werknemer)                    91631
Niet werkend -- niet op zoek naar nieuw werk    28030
Anders                                          17743
Werkend (zelfstandig ondernemer)                 9568
Niet werkend -- op zoek naar werk                6448
Name: count, dtype: int64

In [90]:
df['Werk_binair'] = df['Werk'].apply(lambda x: 1 if x in ['Werkend (betaalde werknemer)', 'Werkend (zelfstandig ondernemer)'] else 0)
# Controleer de unieke waarden
print(df['Werk_binair'].value_counts())

Werk_binair
1    101199
0     52475
Name: count, dtype: int64


In [91]:
df["stance_binary"].value_counts()
df["negative_stance"] = df["stance_binary"].apply(
    lambda x: 1 if str(x).strip().lower() == "negatief" else 0
)

df["negative_stance"].value_counts()

negative_stance
0    132760
1     20914
Name: count, dtype: int64

In [92]:
df['Attention1'].value_counts()

df['Attention_1_fail'] = df['Attention1'].apply(lambda x: 0 if x == 'De Volkskrant' else 1)
df['Attention_2_fail'] = df['Q68'].apply(lambda x: 0 if x == 'Altijd' else 1)

# Controleer de unieke waarden
df['Attention_1_fail'].value_counts()
df['Attention_2_fail'].value_counts()

# Maak een nieuwe kolom 'Both_Attention_Fail' die aangeeft of beide aandachtchecks zijn gemist
df['Both_Attention_Fail'] = ((df['Attention_1_fail'] == 1) & (df['Attention_2_fail'] == 1)).astype(int)

# Controleer de verdeling van de nieuwe kolom
print("\nVerdeling van de 'Both_Attention_Fail' kolom:")
print(df['Both_Attention_Fail'].value_counts())


Verdeling van de 'Both_Attention_Fail' kolom:
Both_Attention_Fail
0    151107
1      2567
Name: count, dtype: int64


In [93]:
df['Edu'].value_counts()

Edu
Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)                             48291
Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)                44874
Wetenschappelijk onderwijs (universiteit)                                                                         19271
Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)                     13964
Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)                                            12743
Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)     7533
Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)                                                     4950
Lagere school (basisonderwijs)                                                                                     1024
Geen onderwijs gevolgd of het niet a

In [94]:
# Functie om de volledige indeling te maken
### Anders, nml is part of 'Basis of Voortgezet onderwijs' becuase the responses here indicated 'Hav' --> havo'''

def recode_edu(value):

    if value in [
        "Geen onderwijs gevolgd of het niet afgemaakt", 
        "Lagere school (basisonderwijs)"
    ]:
        return "Basisonderwijs"
    elif value in [
        "Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)",
        "Anders, namelijk:"
    ]:
        return "Voortgezet onderwijs"
    elif value in [
        "Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)", 
        "Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)", 
        "Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)", 
        "Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)"
    ]:
        return "Praktijkopleiding"
    elif value in [
        "Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)", 
        "Wetenschappelijk onderwijs (universiteit)"
    ]:
        return "Hoger onderwijs"
    # Als er een onverwachte waarde is, raise een fout.
    else:
        raise ValueError(f"Onverwachte waarde: {value}")

# Nieuwe kolom toevoegen met de gerecodeerde waarden
df['Edu_breed'] = df['Edu'].apply(recode_edu)

# Controleer de resultaten
print(df['Edu_breed'].value_counts())

Edu_breed
Praktijkopleiding       70100
Hoger onderwijs         67562
Voortgezet onderwijs    14095
Basisonderwijs           1917
Name: count, dtype: int64


In [95]:
def recode_edu_numeric(value):
    """
    Recodes educational categories into 1 (High) and 0 (Rest).
    'Hoger onderwijs' is classified as '1' (High) and all other categories as '0' (Rest).
    """

    if value in [
        "Geen onderwijs gevolgd of het niet afgemaakt", 
        "Lagere school (basisonderwijs)",
        "Voortgezet algemeen onderwijs (5-jaar HBS, MMS, HAVO, lyceum, atheneum, gymnasium, VWO, etc.)",
        "Anders, namelijk:",
        "Voorbereidend of kort middelbaar beroepsonderwijs (VMBO, KMBO)", 
        "Lager beroepsonderwijs (LBO, LTS, LHNO, huishoud-/ambachts-school, LEAO, lager land-en tuinbouwonderwijs etc.)", 
        "Middelbaar beroepsonderwijs (MBO, MTS, MEAO, Praktijkdiploma Boekhouden, Kleuterkweekschool, etc.)", 
        "Middelbaar algemeen onderwijs (LAVO, ULO, MULO, MAVO, 3-jaar HBS etc.)"
    ]:
        return 0  # Group everything else as 'Rest' (0)
        
    elif value in [
        "Hoger beroepsonderwijs (HBO, HTS, HEAO, Sociale Academie, HHNO, lerarenonderwijsetc.)", 
        "Wetenschappelijk onderwijs (universiteit)"
    ]:
        return 1  # Group 'Hoger onderwijs' as 'High' (1)
    else:
        raise ValueError(f"Unexpected value: {value}")  # Raise error for unexpected values

# Recode the 'Edu_breed' column into numeric values
df['Edu_binary'] = df['Edu'].apply(recode_edu_numeric)

# Check the results
df['Edu_binary'].value_counts()

Edu_binary
0    86112
1    67562
Name: count, dtype: int64

In [96]:
df['PolOrient_1'].value_counts()

PolOrient_1
5.0     27698
7.0     24831
8.0     21828
6.0     20538
4.0     13182
2.0     11995
3.0     10884
9.0      8476
10.0     6111
1.0      5035
0.0      3096
Name: count, dtype: int64

In [97]:
# Functie om politieke oriëntatie te categoriseren
def recode_pol_orientation(value):
    if value <= 3:
        return "Links"
    elif value <= 6:
        return "Midden"
    elif value <= 10:
        return "Rechts"
    else:
        return "Onbekend"  # Dit zou je niet moeten tegenkomen, maar kan worden toegevoegd voor veiligheid.

# Nieuwe kolom toevoegen met de gerecodeerde waarden
df['PolOrientation_cat'] = df['PolOrient_1'].apply(recode_pol_orientation)

# Controleer de resultaten
print(df['PolOrientation_cat'].value_counts())

PolOrientation_cat
Midden    61418
Rechts    61246
Links     31010
Name: count, dtype: int64


In [98]:
df['AttitudeExtr1_1'].value_counts()

AttitudeExtr1_1
Niet eens, maar ook niet oneens    56693
Mee eens                           41607
Mee oneens                         29437
Zeer mee eens                      16849
Zeer mee oneens                     9088
Name: count, dtype: int64

In [99]:
# Functie om de antwoorden om te zetten naar numerieke waarden
def recode_attitude(value):
    if value == "Zeer mee eens":
        return 4
    elif value == "Mee eens":
        return 3
    elif value == "Niet eens, maar ook niet oneens":
        return 2
    elif value == "Mee oneens":
        return 1
    elif value == "Zeer mee oneens":
        return 0
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Functie voor omgekeerde codering (voor AttitudeExtr2_3)
def recode_attitude_reverse(value):
    if value == "Zeer mee eens":
        return 0
    elif value == "Mee eens":
        return 1
    elif value == "Niet eens, maar ook niet oneens":
        return 2
    elif value == "Mee oneens":
        return 3
    elif value == "Zeer mee oneens":
        return 4
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de vier kolommen, met omgekeerde codering voor AttitudeExtr2_3
df['AttitudeExtr2_1_numeric'] = df['AttitudeExtr2_1'].apply(recode_attitude)
df['AttitudeExtr2_2_numeric'] = df['AttitudeExtr2_2'].apply(recode_attitude)
df['AttitudeExtr2_3_numeric'] = df['AttitudeExtr2_3'].apply(recode_attitude_reverse)  # Omgekeerde codering
df['AttitudeExtr2_4_numeric'] = df['AttitudeExtr2_4'].apply(recode_attitude)

# Controleer op ontbrekende waarden
missing_values = df[['AttitudeExtr2_1_numeric', 'AttitudeExtr2_2_numeric', 'AttitudeExtr2_3_numeric', 'AttitudeExtr2_4_numeric']].isnull().sum()
print(f"Ontbrekende waarden per kolom:\n{missing_values}")

# Het samengestelde item maken (bijvoorbeeld het gemiddelde van de vier kolommen)
df['AttitudeExtr2_combined'] = df[['AttitudeExtr2_1_numeric', 'AttitudeExtr2_2_numeric', 'AttitudeExtr2_3_numeric', 'AttitudeExtr2_4_numeric']].mean(axis=1)

# Controleer de beschrijving van het nieuwe samengestelde item
print("\nSamengestelde score beschrijving:")
print(df['AttitudeExtr2_combined'].describe())

Ontbrekende waarden per kolom:
AttitudeExtr2_1_numeric    0
AttitudeExtr2_2_numeric    0
AttitudeExtr2_3_numeric    0
AttitudeExtr2_4_numeric    0
dtype: int64

Samengestelde score beschrijving:
count    153674.000000
mean          2.238524
std           0.934170
min           0.000000
25%           1.500000
50%           2.250000
75%           3.000000
max           4.000000
Name: AttitudeExtr2_combined, dtype: float64


In [100]:
# Functie om de antwoorden om te zetten met omgekeerde codering voor AttitudeExtr1_4
def recode_attitude_reverse(value):
    if value == "Zeer mee eens":
        return 0
    elif value == "Mee eens":
        return 1
    elif value == "Niet eens, maar ook niet oneens":
        return 2
    elif value == "Mee oneens":
        return 3
    elif value == "Zeer mee oneens":
        return 4
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de vier kolommen, met omgekeerde codering voor AttitudeExtr1_4
df['AttitudeExtr1_1_numeric'] = df['AttitudeExtr1_1'].apply(recode_attitude)
df['AttitudeExtr1_2_numeric'] = df['AttitudeExtr1_2'].apply(recode_attitude)
df['AttitudeExtr1_3_numeric'] = df['AttitudeExtr1_3'].apply(recode_attitude)
df['AttitudeExtr1_4_numeric'] = df['AttitudeExtr1_4'].apply(recode_attitude_reverse)  # Omgekeerde codering hier

# Controleer op ontbrekende waarden
missing_values = df[['AttitudeExtr1_1_numeric', 'AttitudeExtr1_2_numeric', 'AttitudeExtr1_3_numeric', 'AttitudeExtr1_4_numeric']].isnull().sum()
print(f"Ontbrekende waarden per kolom:\n{missing_values}")

# Het samengestelde item maken (bijvoorbeeld het gemiddelde van de vier kolommen)
df['AttitudeExtr1_combined'] = df[['AttitudeExtr1_1_numeric', 'AttitudeExtr1_2_numeric', 'AttitudeExtr1_3_numeric', 'AttitudeExtr1_4_numeric']].mean(axis=1)

# Controleer de beschrijving van het nieuwe samengestelde item
print("\nSamengestelde score beschrijving:")
print(df['AttitudeExtr1_combined'].describe())

Ontbrekende waarden per kolom:
AttitudeExtr1_1_numeric    0
AttitudeExtr1_2_numeric    0
AttitudeExtr1_3_numeric    0
AttitudeExtr1_4_numeric    0
dtype: int64

Samengestelde score beschrijving:
count    153674.000000
mean          2.311658
std           0.888020
min           0.000000
25%           1.750000
50%           2.250000
75%           3.000000
max           4.000000
Name: AttitudeExtr1_combined, dtype: float64


In [101]:
import pingouin as pg

# Cronbach's alpha berekenen voor de samengestelde schalen
# Zorg ervoor dat je de numerieke kolommen van de betreffende schalen gebruikt
alpha_1 = pg.cronbach_alpha(data=df[['AttitudeExtr1_1_numeric', 'AttitudeExtr1_2_numeric', 'AttitudeExtr1_3_numeric', 'AttitudeExtr1_4_numeric']])
alpha_2 = pg.cronbach_alpha(data=df[['AttitudeExtr2_1_numeric', 'AttitudeExtr2_2_numeric', 'AttitudeExtr2_3_numeric', 'AttitudeExtr2_4_numeric']])

# Print de resultaten van Cronbach's alpha
print(f"Cronbach's alpha voor AttitudeExtr1 schaal: {alpha_1[0]:.4f}")
print(f"Cronbach's alpha voor AttitudeExtr2 schaal: {alpha_2[0]:.4f}")

Cronbach's alpha voor AttitudeExtr1 schaal: 0.7860
Cronbach's alpha voor AttitudeExtr2 schaal: 0.8160


In [102]:
df[df['variable'] == 'stellingen.political_stance'][['Ideology', 'normalized_party','PolOrient_1']]

df['ImportIssue_1'].value_counts()

# Functie om de antwoorden om te zetten naar numerieke waarden voor ImportIssue_1
def recode_importance(value):
    if value == "Zeer belangrijk":
        return 4
    elif value == "Belangrijk":
        return 3
    elif value == "Niet belangrijk, maar ook niet onbelangrijk":
        return 2
    elif value == "Onbelangrijk":
        return 1
    elif value == "Zeer onbelangrijk":
        return 0
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de kolom ImportIssue_1
df['ImportIssue_1_numeric'] = df['ImportIssue_1'].apply(recode_importance)

# Controleer de beschrijving van de nieuwe numerieke schaal
print("\nSamenvatting van de nieuwe numerieke schaal:")
print(df['ImportIssue_1_numeric'].describe())

# Controleer de waardeverdeling
print("\nVerdeling van de nieuwe numerieke schaal:")
print(df['ImportIssue_1_numeric'].value_counts())



Samenvatting van de nieuwe numerieke schaal:
count    153674.000000
mean          3.015780
std           0.806659
min           0.000000
25%           3.000000
50%           3.000000
75%           4.000000
max           4.000000
Name: ImportIssue_1_numeric, dtype: float64

Verdeling van de nieuwe numerieke schaal:
ImportIssue_1_numeric
3    81152
4    41358
2    25497
1     3565
0     2102
Name: count, dtype: int64


In [103]:
df['KnowIssue_1'].value_counts()

KnowIssue_1
Niet veel, maar ook niet weinig    80226
Veel                               52066
Weinig                             10813
Heel veel                           8764
Heel weinig                         1805
Name: count, dtype: int64

In [104]:
# Functie om de antwoorden om te zetten naar numerieke waarden voor KnowIssue_1
def recode_knowledge(value):
    if value == "Heel veel":
        return 4
    elif value == "Veel":
        return 3
    elif value == "Niet veel, maar ook niet weinig":
        return 2
    elif value == "Weinig":
        return 1
    elif value == "Heel weinig":
        return 0
    else:
        return None  # Voor eventuele ontbrekende of ongebruikelijke waarden

# Pas de recodering toe op de kolom KnowIssue_1
df['KnowIssue_1_numeric'] = df['KnowIssue_1'].apply(recode_knowledge)

# Controleer de beschrijving van de nieuwe numerieke schaal
print("\nSamenvatting van de nieuwe numerieke schaal:")
print(df['KnowIssue_1_numeric'].describe())

# Controleer de waardeverdeling
print("\nVerdeling van de nieuwe numerieke schaal:")
print(df['KnowIssue_1_numeric'].value_counts())


Samenvatting van de nieuwe numerieke schaal:
count    153674.000000
mean          2.359013
std           0.745242
min           0.000000
25%           2.000000
50%           2.000000
75%           3.000000
max           4.000000
Name: KnowIssue_1_numeric, dtype: float64

Verdeling van de nieuwe numerieke schaal:
KnowIssue_1_numeric
2    80226
3    52066
1    10813
4     8764
0     1805
Name: count, dtype: int64


In [105]:
df[df['variable'] == 'stellingen.political_stance'][['Ideology', 'normalized_party','PolOrient_1']]

df['instruction_type'].value_counts()
df['source_shown'].value_counts()

source_shown
0    77664
1    76010
Name: count, dtype: int64

In [106]:
df['variable'].value_counts()

variable
stellingen.misinformation      34811
stellingen.toxic               34809
stellingen.political_stance    34809
stellingen.sentiment           34808
confirm                         3980
political_stance_training       2642
sentiment_training              2618
toxic_training                  2603
misinformation_training         2594
Name: count, dtype: int64

In [107]:
# Encode the stance as an ordinal variable
scale_mapping = {
    "Helemaal niet waar": 1,
    "Gedeeltelijk niet waar": 2,
    "Neutraal": 3,
    "Gedeeltelijk waar": 4,
    "Helemaal waar": 5
}

df['value_nan'] = df['value'].replace(['Waar', 'Niet waar', 'confirmed'], np.nan)

# Map the remaining categories to a numeric scale
df['value_scaled'] = df['value_nan'].map(scale_mapping)
# Check the updated DataFrame
print(df[['value', 'value_scaled']].tail())

                         value  value_scaled
107929                Neutraal           3.0
107942       Gedeeltelijk waar           4.0
107943  Gedeeltelijk niet waar           2.0
107944       Gedeeltelijk waar           4.0
107945                Neutraal           3.0


In [108]:
# Check for missing values in key columns
df[df['variable'] == 'stellingen.toxic'][['vote_likelihood_score', 'source_shown', 'dsc', 'instruction_type', 'value_scaled']].isna().sum()

vote_likelihood_score    0
source_shown             0
dsc                      0
instruction_type         0
value_scaled             0
dtype: int64

In [109]:
df['variable'].value_counts()

variable
stellingen.misinformation      34811
stellingen.toxic               34809
stellingen.political_stance    34809
stellingen.sentiment           34808
confirm                         3980
political_stance_training       2642
sentiment_training              2618
toxic_training                  2603
misinformation_training         2594
Name: count, dtype: int64

## Saving the DF

In [111]:
# Convert 'tweet_id' to string (if needed)
df['tweet_id_str'] = df['tweet_id'].astype(str)

# Define output path
output_dir = "/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/"
filename_base = "final_merged_dataset_for_analysis"

# Save as CSV with proper quoting
df.to_csv(f"{output_dir}/{filename_base}.csv", index=False, quoting=1)

# Save as Parquet (preserves data types)
df.to_parquet(f"{output_dir}/{filename_base}.parquet", index=False)

# Save as Pickle (efficient for Python reloading)
df.to_pickle(f"{output_dir}/{filename_base}.pkl")

In [13]:
df = pd.read_parquet("/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/final_merged_dataset_for_analysis.parquet")
# Print original number of rows 

In [6]:
n_participants = df['uid'].nunique()

# Basic demographic fields available
demographic_fields = [
    'Age', 'Gender', 'Etniciteit_binair', 'Edu_breed', 'PolOrientation_cat',
    'dsc', 'vote_likelihood_score', 'instruction_type'
]

# Check how many participants have non-null values for each key demographic variable
demographics_summary = df.groupby('uid')[demographic_fields].first().describe(include='all')

n_participants, demographics_summary.T[['count', 'mean', 'std', 'min', 'max']]

(1358,
                         count       mean        std       min       max
 Age                    1358.0  51.363034  15.718696      25.0      90.0
 Gender                   1358        NaN        NaN       NaN       NaN
 Etniciteit_binair      1358.0   0.918999   0.272938       0.0       1.0
 Edu_breed                1358        NaN        NaN       NaN       NaN
 PolOrientation_cat       1358        NaN        NaN       NaN       NaN
 dsc                    1358.0   0.515095   0.447804 -1.288011  1.528017
 vote_likelihood_score  1248.0   2.111378   1.230304       1.0       5.0
 instruction_type         1358        NaN        NaN       NaN       NaN)

In [1]:
## AFter removing attention check questions

In [14]:
import pandas as pd

df = pd.read_parquet("/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/final_merged_dataset_for_analysis.parquet")
# Print original number of rows 

#### Creating samples with strickter QC settings for robustness checks 

In [15]:
print("Before filtering:", len(df))

# Remove rows where any attention check failed
df = df[
    (df['Attention_1_fail'] != True) &
    (df['Attention_2_fail'] != True) &
    (df['Both_Attention_Fail'] != True)
].copy()

# Print number of rows after filtering
print("After filtering:", len(df))

# Sanity check: confirm no failures remain
print("\nRemaining Attention_1_fail counts:")
print(df['Attention_1_fail'].value_counts())

print("\nRemaining Attention_2_fail counts:")
print(df['Attention_2_fail'].value_counts())

print("\nRemaining Both_Attention_Fail counts:")
print(df['Both_Attention_Fail'].value_counts())

Before filtering: 153674
After filtering: 143100

Remaining Attention_1_fail counts:
Attention_1_fail
0    143100
Name: count, dtype: int64

Remaining Attention_2_fail counts:
Attention_2_fail
0    143100
Name: count, dtype: int64

Remaining Both_Attention_Fail counts:
Both_Attention_Fail
0    143100
Name: count, dtype: int64


In [16]:
n_participants = df['uid'].nunique()

# Basic demographic fields available
demographic_fields = [
    'Age', 'Gender', 'Etniciteit_binair', 'Edu_breed', 'PolOrientation_cat',
    'dsc', 'vote_likelihood_score', 'instruction_type'
]

# Check how many participants have non-null values for each key demographic variable
demographics_summary = df.groupby('uid')[demographic_fields].first().describe(include='all')

n_participants, demographics_summary.T[['count', 'mean', 'std', 'min', 'max']]

(1270,
                         count      mean        std       min       max
 Age                    1270.0  52.06378  15.617922      25.0      90.0
 Gender                   1270       NaN        NaN       NaN       NaN
 Etniciteit_binair      1270.0  0.922047   0.268203       0.0       1.0
 Edu_breed                1270       NaN        NaN       NaN       NaN
 PolOrientation_cat       1270       NaN        NaN       NaN       NaN
 dsc                    1270.0  0.530691   0.441002 -1.192527  1.528017
 vote_likelihood_score  1163.0  2.066208   1.220134       1.0       5.0
 instruction_type         1270       NaN        NaN       NaN       NaN)

In [17]:
# Define output path
output_dir = "/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/"
filename_base = "final_merged_dataset_for_analysis_without_failing_attention_check"

# Save as CSV with proper quoting
df.to_csv(f"{output_dir}/{filename_base}.csv", index=False, quoting=1)

# Save as Parquet (preserves data types)
df.to_parquet(f"{output_dir}/{filename_base}.parquet", index=False)

# Save as Pickle (efficient for Python reloading)
df.to_pickle(f"{output_dir}/{filename_base}.pkl")

In [7]:
# ---------------------------
# QC PIPELINE FOR CROWDCODING (with structural straightliners) + DROP LIST
# ---------------------------
import pandas as pd
import numpy as np
from dataclasses import dataclass

# ==== CONFIG ====
ANNOTATOR_COL = "uid"                 # <-- your annotator column
ANSWER_COL    = "value_scaled"        # <-- numeric label column (1..5)

ANSWER_MAP = {  # leave empty if already numeric
    # "Strongly disagree": 1, "Disagree": 2, "Neutral": 3, "Agree": 4, "Strongly agree": 5
}

VARIABLES_OF_INTEREST = {
    "stellingen.misinformation",
    "stellingen.sentiment",
    "stellingen.toxic",
}

@dataclass
class QCConfig:
    # item-level timing
    min_rt_item_s: float = 0.8
    # annotator-level timing
    median_rt_floor_s: float = 1.0
    # straightlining ingredients
    longstring_k: int = 15         # longest identical run threshold (per outcome)
    sd_floor: float = 0.25         # low variance threshold (per outcome)
    entropy_floor_bits: float = 0.9# low entropy threshold (per outcome)
    # other optional flags
    edge_prop_thresh: float = 0.90
    repeat_tol: float = 1.0
    # calibration / honeypots
    cali_dev_tol: float = 1.0
    cali_miss_allow: int = 1
    cali_min_items: int = 3
    consensus_item_entropy_bits: float = 0.5

cfg = QCConfig()

# ==== HELPERS ====
def coerce_answer(series: pd.Series) -> pd.Series:
    if ANSWER_MAP:
        series = series.map(ANSWER_MAP).astype("float")
    else:
        series = pd.to_numeric(series, errors="coerce")
    return series

def shannon_entropy(counts) -> float:
    counts = np.array(counts, dtype=float)
    if counts.sum() == 0:
        return np.nan
    p = counts[counts > 0] / counts.sum()
    return float(-(p * np.log2(p)).sum())

def longest_run(values: pd.Series) -> int:
    vals = values.values
    best = cur = 0
    prev = object()
    for v in vals:
        if pd.isna(v):
            cur = 0; prev = object(); continue
        if v == prev:
            cur += 1
        else:
            cur = 1
        prev = v
        if cur > best:
            best = cur
    return int(best)

def item_entropy(series: pd.Series) -> float:
    vc = series.value_counts(dropna=True)
    return shannon_entropy(vc.values)

def safe_quantile(s: pd.Series, q):
    try:
        return float(s.quantile(q))
    except Exception:
        return np.nan

# ==== PREP ====
def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    # keep variables of interest (plus any training items)
    mask = df["variable"].isin(VARIABLES_OF_INTEREST) | df["variable"].str.contains("training", case=False, na=False)
    df = df.loc[mask].copy()
    if "unit_status" in df.columns:
        df = df.loc[df["unit_status"].eq("DONE")].copy()

    # times and response time
    df["t_q"] = pd.to_datetime(df["time_question_annotinder"], utc=True, errors="coerce")
    df["t_a"] = pd.to_datetime(df["time_answer"], utc=True, errors="coerce")
    df["rt_s"] = (df["t_a"] - df["t_q"]).dt.total_seconds()
    df["rt_fast_flag"] = df["rt_s"] < cfg.min_rt_item_s

    # answers -> numeric
    df[ANSWER_COL] = coerce_answer(df[ANSWER_COL])

    # endpoints for edge proportion
    label_min = int(np.nanmin(df[ANSWER_COL].values))
    label_max = int(np.nanmax(df[ANSWER_COL].values))
    df["_label_min"] = label_min
    df["_label_max"] = label_max
    return df

# ==== CALIBRATION / HONEYPOT ====
def build_calibration_table(df: pd.DataFrame) -> pd.DataFrame:
    train_mask = df["variable"].str.contains("training", case=False, na=False)
    if train_mask.any():
        base = df.loc[train_mask].copy()
        src = "training"
    else:
        base = df.loc[df["variable"].isin(VARIABLES_OF_INTEREST)].copy()
        ent = (base.groupby(["variable", "unit_id"])[ANSWER_COL]
                    .apply(item_entropy)
                    .rename("entropy"))
        base = base.merge(ent, on=["variable", "unit_id"], how="left")
        base = base.loc[base["entropy"] <= cfg.consensus_item_entropy_bits].copy()
        src = "consensus-fallback"

    if base.empty:
        return pd.DataFrame(columns=["variable", "unit_id", "consensus", "n", "source"])

    def consensus_func(s):
        vc = s.value_counts()
        if vc.empty: return np.nan
        top = vc.max(); modes = vc[vc == top].index.to_list()
        return float(modes[0]) if len(modes) == 1 else float(np.mean(modes))

    tab = (base.groupby(["variable", "unit_id"])
                .agg(consensus=(ANSWER_COL, consensus_func), n=(ANSWER_COL, "size"))
                .reset_index())
    tab["source"] = src
    return tab

# ==== REPEATED ITEMS ====
def per_annotator_repeat_inconsistency(df: pd.DataFrame) -> pd.Series:
    rep = df.groupby([ANNOTATOR_COL, "variable", "unit_id"]).size().rename("n").reset_index()
    rep = rep.loc[rep["n"] >= 2, [ANNOTATOR_COL, "variable", "unit_id"]]
    if rep.empty:
        return pd.Series(dtype=float, name="repeat_mad")
    merged = df.merge(rep, on=[ANNOTATOR_COL, "variable", "unit_id"], how="inner")

    def mad_pairs(s):
        v = s.dropna().values
        if v.size < 2: return np.nan
        diffs = [abs(v[i]-v[j]) for i in range(len(v)) for j in range(i+1, len(v))]
        return float(np.median(diffs)) if diffs else np.nan

    out = (merged.groupby([ANNOTATOR_COL, "variable"])[ANSWER_COL]
                 .apply(mad_pairs)
                 .groupby(level=0).median()
                 .rename("repeat_mad"))
    return out

# ==== MAIN QC ====
def run_qc(df_raw: pd.DataFrame, cfg: QCConfig = cfg):
    df = prepare_df(df_raw)

    # ---- per-annotator aggregates ----
    g_anno = df.groupby(ANNOTATOR_COL, as_index=False)
    speed = g_anno.agg(
        n_items=("unit_id", "size"),
        median_rt=("rt_s", "median"),
        p01_rt=("rt_s", lambda s: safe_quantile(s, 0.01)),
        p99_rt=("rt_s", lambda s: safe_quantile(s, 0.99)),
        fast_click_rate=("rt_fast_flag", "mean"),
    )
    speed["speed_flag"] = speed["median_rt"] < cfg.median_rt_floor_s

    # per-outcome stats, then worst-case per annotator
    label_min = int(df["_label_min"].iloc[0]); label_max = int(df["_label_max"].iloc[0])
    def per_outcome_stats(d):
        d = d.sort_values("t_q")
        length = longest_run(d[ANSWER_COL])
        sd = float(d[ANSWER_COL].std(ddof=0))
        ent = item_entropy(d[ANSWER_COL])
        edge_prop = float(((d[ANSWER_COL] == label_min) | (d[ANSWER_COL] == label_max)).mean())
        return pd.Series({"longstring": length, "sd": sd, "entropy": ent, "edge_prop": edge_prop})

    stats = (df.groupby([ANNOTATOR_COL, "variable"])
               .apply(per_outcome_stats)
               .reset_index())

    worst = (stats.groupby(ANNOTATOR_COL)
                  .agg(longstring_max=("longstring", "max"),
                       sd_min=("sd", "min"),
                       entropy_min=("entropy", "min"),
                       edge_prop_max=("edge_prop", "max"))
                  .reset_index())

    # base flags
    worst["longstring_flag"] = worst["longstring_max"] >= cfg.longstring_k
    worst["lowvar_flag"]     = worst["sd_min"] < cfg.sd_floor
    worst["lowent_flag"]     = worst["entropy_min"] < cfg.entropy_floor_bits
    worst["edge_flag"]       = worst["edge_prop_max"] > cfg.edge_prop_thresh

    # *** STRUCTURAL STRAIGHTLINER ***
    # low entropy AND (longstring OR low variance)
    worst["struct_straightliner_flag"] = worst["lowent_flag"] & (worst["longstring_flag"] | worst["lowvar_flag"])

    # repeat consistency
    repeat = per_annotator_repeat_inconsistency(df)
    repeat = repeat.reset_index() if isinstance(repeat, pd.Series) else repeat
    if repeat.empty:
        repeat = pd.DataFrame({ANNOTATOR_COL: df[ANNOTATOR_COL].unique(), "repeat_mad": np.nan})
    repeat["repeat_flag"] = repeat["repeat_mad"] > cfg.repeat_tol

    # calibration/honeypots
    calitab = build_calibration_table(df)
    if not calitab.empty:
        joined = df.merge(calitab, on=["variable", "unit_id"], how="inner")
        joined["cali_miss"] = (joined[ANSWER_COL] - joined["consensus"]).abs() > cfg.cali_dev_tol
        cali = (joined.groupby(ANNOTATOR_COL)["cali_miss"]
                      .agg(["sum","size"])
                      .rename(columns={"sum":"cali_misses","size":"cali_n"})
                      .reset_index())
        cali["cali_flag"] = (cali["cali_n"] >= cfg.cali_min_items) & (cali["cali_misses"] > cfg.cali_miss_allow)
    else:
        cali = pd.DataFrame({ANNOTATOR_COL: df[ANNOTATOR_COL].unique(),
                             "cali_misses": np.nan, "cali_n": 0, "cali_flag": False})

    # merge diagnostics
    qc = (speed.merge(worst, on=ANNOTATOR_COL, how="left")
                .merge(repeat, on=ANNOTATOR_COL, how="left")
                .merge(cali, on=ANNOTATOR_COL, how="left"))

    # ----- GUARDED FLAG COUNT -----
    # Use structural straightliner flag (instead of raw low-entropy) in totals
    base_flags = ["speed_flag", "longstring_flag", "lowvar_flag", "edge_flag", "repeat_flag", "cali_flag"]
    qc["n_flags"] = qc[base_flags].sum(axis=1, numeric_only=True) + qc["struct_straightliner_flag"].astype(int)

    def action_row(r):
        # Drop only with converging evidence:
        #  - n_flags >= 3 (structural straightliner counts as one), OR
        #  - calibration AND speed flags together
        if (r["n_flags"] >= 3) or (r.get("cali_flag", False) and r.get("speed_flag", False)):
            return "drop"
        if r["n_flags"] >= 1:
            return "weight"
        return "keep"

    qc["action"] = qc.apply(action_row, axis=1)

    # reliability weights (for models) based on guarded flags
    qc["weight"] = (1.0 - 0.2 * qc["n_flags"]).clip(lower=0.30, upper=1.00)

    # merge per-annotation
    df_qc = df.merge(qc[[ANNOTATOR_COL, "weight", "action", "struct_straightliner_flag"]], on=ANNOTATOR_COL, how="left")

    # --- OUTPUTS: lists of UIDs to remove ---
    DROP_UIDS = (qc.loc[qc["action"]=="drop", ANNOTATOR_COL].dropna().sort_values().unique().tolist())
    DROP_STRUCT_UIDS = (qc.query("action == 'drop' and struct_straightliner_flag")[ANNOTATOR_COL]
                          .dropna().sort_values().unique().tolist())
    # If you ever want "all structural straightliners" regardless of action:
    ALL_STRUCT_UIDS = (qc.loc[qc["struct_straightliner_flag"], ANNOTATOR_COL]
                         .dropna().sort_values().unique().tolist())

    # cleaned data for modeling (keeps weighted raters, removes 'drop')
    df_qc_clean = df_qc[df_qc["action"] != "drop"].copy()

    # quick prints
    print(f"Annotators total: {qc.shape[0]}")
    print(f"Dropped annotators (all reasons): {len(DROP_UIDS)}")
    print(f"Dropped annotators that are structural straightliners: {len(DROP_STRUCT_UIDS)}")
    print(f"Kept annotations for modeling: {df_qc_clean.shape[0]} / {df_qc.shape[0]} ({df_qc_clean.shape[0]/max(1,df_qc.shape[0]):.1%})")

    # Optional: save audit CSVs
    # pd.Series(DROP_UIDS, name=ANNOTATOR_COL).to_csv("drop_annotators.csv", index=False)
    # qc.to_csv("qc_summary.csv", index=False)

    return df_qc, qc, df_qc_clean, DROP_UIDS, DROP_STRUCT_UIDS, ALL_STRUCT_UIDS, stats, calitab

# === RUN ===
# df should contain: ['time_question_annotinder','time_answer','variable','unit_status','unit_id','uid','value_scaled', ...]
df_qc, qc_summary, df_qc_clean, DROP_UIDS, DROP_STRUCT_UIDS, ALL_STRUCT_UIDS, outcome_stats, calib_table = run_qc(df, cfg)

# Example: filter original data or QC-augmented data
# df_clean = df[~df[ANNOTATOR_COL].isin(DROP_UIDS)].copy()
# df_qc_clean already excludes 'drop' and includes weights.

/tmp/ipykernel_45376/2439071534.py:184: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(per_outcome_stats)


Annotators total: 1232
Dropped annotators (all reasons): 141
Dropped annotators that are structural straightliners: 141
Kept annotations for modeling: 94349 / 106898 (88.3%)


``` 
How straightliners were removed (text for your Methods/QC section)

We identified structural straightliners at the annotator level using an order-aware and distribution-aware rule. For each outcome (misinformation, sentiment, toxicity) and annotator we computed: (a) the longest run of identical responses (“longstring”), (b) the within-outcome response variance, and (c) the response entropy. An annotator was flagged as a structural straightliner if they showed low entropy (≤ 0.9 bits) AND (longstring ≥ 15 identical responses in a row OR SD < 0.25). This ensures we only flag monotone responding when it is both low-diversity overall and consistent with straightlining behavior over time, protecting annotators who simply faced skewed item sets.

Annotators were removed (action = drop) only under converging evidence: when the total number of QC flags (including the structural-straightliner flag, speed, longstring, low variance, edge-use, repeats, and calibration) was ≥ 3, or when both calibration and speed flags were triggered. Otherwise, annotators were retained and down-weighted according to their flag count (weight = 1 − 0.2·flags, clipped to [0.30, 1.00]). This conservative approach avoids “cleaning away” substantive perspective differences while removing clear cases of response set behavior.

```
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [11]:
len(df_qc_clean['uid'].unique())

1091

In [12]:
# Define output path
output_dir = "/home/akroon/webdav/ASCOR-FMG-4394-AnNoBias (Projectfolder)/data/"
filename_base = "final_merged_dataset_for_analysis_without_failing_attention_check_without_straightliners"

# Save as CSV with proper quoting
df.to_csv(f"{output_dir}/{filename_base}.csv", index=False, quoting=1)

# Save as Parquet (preserves data types)
df.to_parquet(f"{output_dir}/{filename_base}.parquet", index=False)

# Save as Pickle (efficient for Python reloading)
df.to_pickle(f"{output_dir}/{filename_base}.pkl")